# Expected points with intervals for one situation

This notebook picks up where `epa_calc.ipynb` ends. Enter a 4th-down situation and it prints what `epa_calc`'s `evaluate()` prints, with an interval on each number.

**How to use it:** run sections 0–2 once (about 2 minutes). Then change the situation at the top of section 3 and rerun just that cell, which takes a couple of seconds.

**How the intervals are made:** section 2 draws the training games with replacement 1,000 times (whole games, not single plays, because plays in the same game share teams, weather and game script). For each resample it rebuilds the lookup tables with `epa_calc`'s own code. Section 3 evaluates the situation with `epa_calc`'s tables (the estimate) and with each resample's tables (the interval).

**What the intervals cover:** how much the results depend on which games happen to be in the data. They are narrow partly because the model pools data. For example, the 4th-and-1 conversion rate pools every 3rd- and 4th-and-1 attempt by every team at every spot. So they measure uncertainty from limited data, not the full uncertainty of a real decision, which also depends on the teams and the play. They also leave out error in nflverse's own EP values and the model's simplifications.

## 0. Run `epa_calc.ipynb`

This runs `epa_calc.ipynb` (kept in the same folder) in this notebook, so its data, tables and `evaluate_frame()` are available here. Its printouts are hidden; run `epa_output.show()` to see them. You can instead paste the cells below at the end of `epa_calc.ipynb` and skip this one.

In [1]:
# %%capture epa_output
# %run ./epa_calc.ipynb

In [2]:
import numpy as np
import nflreadpy as nfl
import pandas as pd
import time
import warnings

warnings.filterwarnings("ignore", message="invalid value encountered in divide")

%run -i evaluate_frame.py # import evaluate_frame

In [3]:
TRAIN_SEASONS = list(range(2021, 2026))   # select seasons
EVAL_SEASON = 2025                        # test seasons
MAX_YTG = 15                              # 15+ yards to go are pooled
MIN_TIME_TO_SCORE = 30                    # seconds needed after getting the ball back
END_OF_HALF_SECS = 5                      # time past which is the last play of the half
TD_POINTS = 7.0
FG_POINTS = 3.0
 
COLS = [
    "season", "week", "game_id", "posteam", "defteam", "qtr", "down", "ydstogo",
    "yardline_100", "play_type", "ep", "epa", "yards_gained", "fumble_lost",
    "interception", "penalty", "qb_kneel", "qb_spike", "kick_distance",
    "field_goal_result", "game_seconds_remaining", "score_differential",
    "posteam_timeouts_remaining", "desc",
]
 
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
 
pbp = nfl.load_pbp(seasons=TRAIN_SEASONS).select(COLS).to_pandas()
clean = (pbp["qb_kneel"].fillna(0) == 0) & (pbp["qb_spike"].fillna(0) == 0)

In [4]:
# Load data from epa_calc
EXPORT_FILE = "epa_calc_export.npz"
saved = np.load(EXPORT_FILE)

MAX_YTG = saved["MAX_YTG"].item()
TD_POINTS = saved["TD_POINTS"].item()
FG_POINTS = saved["FG_POINTS"].item()
MIN_TIME_TO_SCORE = saved["MIN_TIME_TO_SCORE"].item()
END_OF_HALF_SECS = saved["END_OF_HALF_SECS"].item()
TRAIN_SEASONS = saved["TRAIN_SEASONS"].tolist()

# arrays from the field-goal cell
yards = saved["yards"]
miss_opp = saved["miss_opp"]

# lookup tables from epa_calc
EP1 = saved["EP1"]
P_CONVERT = saved["P_CONVERT"]
SUCCESS = saved["SUCCESS"]
TD_SHARE = saved["TD_SHARE"]
PUNT = saved["PUNT"]
P_FG = saved["P_FG"]
FG_EP = saved["FG_EP"]

TABLE_NAMES = ["EP1", "P_CONVERT", "SUCCESS", "TD_SHARE", "PUNT", "P_FG", "FG_EP"]

## Bootstrap the lookup tables

In [5]:
# Settings
N_BOOT = 1000   # number of game resamples
SEED = 2026
LEVEL = 0.95

TABLE_NAMES = ["EP1", "P_CONVERT", "SUCCESS", "TD_SHARE", "PUNT", "P_FG", "FG_EP"]

In [6]:
train = pbp[[
    "game_id", "down", "ydstogo", "yardline_100", "play_type", "ep", "epa", "yards_gained",
    "fumble_lost", "interception", "penalty", "qb_kneel", "qb_spike", "kick_distance",
    "field_goal_result",
]].reset_index(drop=True)

game_code, game_ids = pd.factorize(train["game_id"])
n_games = len(game_ids)
print(f"Training data: {n_games:,} games, {len(train):,} plays ({TRAIN_SEASONS[0]}-{TRAIN_SEASONS[-1]})")

point_tables = {"EP1": EP1, "P_CONVERT": P_CONVERT, "SUCCESS": SUCCESS, "TD_SHARE": TD_SHARE,
                "PUNT": PUNT, "P_FG": P_FG, "FG_EP": FG_EP}

rng = np.random.default_rng(SEED)
boot_tables = []
t0 = time.time()
try:
    for b in range(N_BOOT):
        # 1. Resample games
        drawn = np.bincount(rng.integers(0, n_games, n_games), minlength=n_games)
        bp = train.iloc[np.repeat(np.arange(len(train)), drawn[game_code])]
        b_clean = (bp["qb_kneel"].fillna(0) == 0) & (bp["qb_spike"].fillna(0) == 0)

        # 2. Rebuild the lookup tables
        # EP of having 1st down at each yard line
        b_first = bp[(bp["down"] == 1) & bp["ep"].notna() & bp["yardline_100"].between(1, 99)]
        b_ep1 = (
            b_first.groupby(b_first["yardline_100"].astype(int))["ep"].mean()
            .reindex(range(1, 100))
            .interpolate(limit_direction="both")
            .rolling(5, center=True, min_periods=1).mean()
        )
        EP1 = np.zeros(101)
        EP1[1:100] = b_ep1.to_numpy()

        # Conversion rate by yards to go - 3rd + 4th down run/pass plays
        b_att = bp[
            bp["down"].isin([3, 4])
            & bp["play_type"].isin(["run", "pass"])
            & b_clean
            & (bp["penalty"].fillna(0) == 0)
            & bp["ydstogo"].notna()
            & bp["yards_gained"].notna()
        ].copy()
        b_att["ytg"] = b_att["ydstogo"].astype(int).clip(1, MAX_YTG)
        b_att["converted"] = (
            (b_att["yards_gained"] >= b_att["ydstogo"])
            & (b_att["fumble_lost"].fillna(0) == 0)
            & (b_att["interception"].fillna(0) == 0)
        )
        b_conv = (
            b_att.groupby("ytg")["converted"].mean()
            .reindex(range(1, MAX_YTG + 1))
            .interpolate(limit_direction="both")
        )
        P_CONVERT = np.zeros(MAX_YTG + 1)
        P_CONVERT[1:] = np.minimum.accumulate(b_conv.to_numpy())

        # EP after a successful conversion - real yards gained on conversions
        SUCCESS = np.full((100, MAX_YTG + 1), np.nan)
        TD_SHARE = np.zeros((100, MAX_YTG + 1))
        for k in range(1, MAX_YTG + 1):
            b_gains = b_att.loc[b_att["converted"] & (b_att["ytg"] == k), "yards_gained"].astype(int).to_numpy()
            for y in range(k, 100):
                b_spot = y - b_gains
                SUCCESS[y, k] = np.where(b_spot <= 0, TD_POINTS, EP1[np.clip(b_spot, 1, 99)]).mean()
                TD_SHARE[y, k] = (b_spot <= 0).mean()

        # Punt value by yard line
        b_punts = bp[(bp["play_type"] == "punt") & bp["ep"].notna() & bp["epa"].notna()].copy()
        b_punts["y"] = b_punts["yardline_100"].astype(int)
        b_punts["ep_after"] = b_punts["ep"] + b_punts["epa"]
        b_pg = b_punts.groupby("y")["ep_after"].agg(["mean", "count"]).reindex(range(1, 100))
        b_punt_ok = b_pg["count"].fillna(0) >= 20
        b_punt_ep = (
            b_pg["mean"].where(b_punt_ok)
            .rolling(5, center=True, min_periods=1).mean()
            .where(b_punt_ok)
        )
        PUNT = np.full(100, np.nan)
        PUNT[1:] = b_punt_ep.to_numpy()

        # Field goal make rate by kick distance
        b_fgs = bp[(bp["play_type"] == "field_goal") & bp["kick_distance"].notna()].copy()
        b_fgs["dist"] = b_fgs["kick_distance"].astype(int)
        b_fgs["made"] = (b_fgs["field_goal_result"] == "made").astype(int)
        b_fd = b_fgs.groupby("dist")["made"].agg(["sum", "count"]).reindex(range(18, 80), fill_value=0)
        b_made_roll = b_fd["sum"].rolling(5, center=True, min_periods=1).sum()
        b_att_roll = b_fd["count"].rolling(5, center=True, min_periods=1).sum()
        b_fg_rate = (b_made_roll / b_att_roll).where(b_att_roll >= 10).bfill().fillna(0).cummin()
        P_FG = np.array([b_fg_rate.get(y + 17, 0.0) for y in yards])  # snap 7 + end zone 10
        FG_EP = np.where(P_FG > 0, P_FG * FG_POINTS + (1 - P_FG) * -EP1[miss_opp], np.nan)

        # 3. Keep tables from resample
        boot_tables.append({"EP1": EP1, "P_CONVERT": P_CONVERT, "SUCCESS": SUCCESS,
                            "TD_SHARE": TD_SHARE, "PUNT": PUNT, "P_FG": P_FG, "FG_EP": FG_EP})

        if (b + 1) % 100 == 0:
            print(f"{b + 1:,}/{N_BOOT:,} resamples ({time.time() - t0:.0f}s)")
finally:
    EP1, P_CONVERT, SUCCESS, TD_SHARE, PUNT, P_FG, FG_EP = (point_tables[k] for k in TABLE_NAMES)

print(f"Done: {len(boot_tables):,} sets of tables in {time.time() - t0:.0f}s")

Training data: 1,424 games, 247,284 plays (2021-2025)
100/1,000 resamples (6s)
200/1,000 resamples (12s)
300/1,000 resamples (18s)
400/1,000 resamples (24s)
500/1,000 resamples (30s)
600/1,000 resamples (36s)
700/1,000 resamples (42s)
800/1,000 resamples (49s)
900/1,000 resamples (55s)
1,000/1,000 resamples (62s)
Done: 1,000 sets of tables in 62s


In [7]:
BOOT_FILE = "epa_bootstrap_tables.npz"

np.savez_compressed(
    BOOT_FILE,
    # settings for evaluate_frame()
    MAX_YTG=MAX_YTG, TD_POINTS=TD_POINTS, FG_POINTS=FG_POINTS,
    MIN_TIME_TO_SCORE=MIN_TIME_TO_SCORE, END_OF_HALF_SECS=END_OF_HALF_SECS,
    TABLE_NAMES=TABLE_NAMES,
    # Original tables from epa_calc
    **{f"point_{k}": point_tables[k] for k in TABLE_NAMES},
    # Tables from all resamples
    **{f"boot_{k}": np.stack([T[k] for T in boot_tables]) for k in TABLE_NAMES},
)
print(f"Saved {len(boot_tables):,} resamples to {BOOT_FILE}")

Saved 1,000 resamples to epa_bootstrap_tables.npz
